# S&P 500 Time Series Forecasting

This notebook demonstrates various time series forecasting methods applied to S&P 500 stock prices.

## Methods Covered:
1. Moving Average
2. Linear Regression
3. ARIMA (AutoRegressive Integrated Moving Average)
4. SARIMAX (Seasonal ARIMA with eXogenous factors)
5. Exponential Smoothing

## Workflow:
1. Data Loading
2. Exploratory Data Analysis
3. Data Preprocessing
4. Model Training and Forecasting
5. Model Evaluation and Comparison

## 1. Import Libraries

In [ ]:
import sys
import os

# Add src directory to path
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Import custom modules
from data_loader import fetch_sp500_data, save_data, load_data, get_close_prices
from preprocessing import (
    check_missing_values, fill_missing_values, 
    train_test_split_timeseries, normalize_data,
    create_lag_features, create_rolling_features
)
from models import (
    ARIMAModel, SARIMAXModel, ExponentialSmoothingModel,
    MovingAverageModel, LinearRegressionModel
)
from evaluation import (
    evaluate_forecast, print_evaluation_results, compare_models
)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 2. Load S&P 500 Data

In [ ]:
# Fetch S&P 500 data for the last 5 years
print("Fetching S&P 500 data...")
sp500_data = fetch_sp500_data(period="5y")

# Display basic information
print(f"\nData shape: {sp500_data.shape}")
print(f"Date range: {sp500_data.index[0]} to {sp500_data.index[-1]}")
print(f"\nColumns: {list(sp500_data.columns)}")

# Display first few rows
sp500_data.head()

In [ ]:
# Save data for future use
save_data(sp500_data, '../data/sp500_data.csv')

## 3. Exploratory Data Analysis

In [ ]:
# Check for missing values
missing_values = check_missing_values(sp500_data)
print("Missing values per column:")
print(missing_values)

In [ ]:
# Get descriptive statistics
sp500_data.describe()

In [ ]:
# Plot closing prices over time
plt.figure(figsize=(15, 6))
plt.plot(sp500_data.index, sp500_data['Close'], linewidth=1.5)
plt.title('S&P 500 Closing Prices Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Close Price (USD)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot volume traded
plt.figure(figsize=(15, 6))
plt.plot(sp500_data.index, sp500_data['Volume'], linewidth=1, alpha=0.7)
plt.title('S&P 500 Trading Volume Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Volume', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot daily returns
sp500_data['Daily_Return'] = sp500_data['Close'].pct_change() * 100

plt.figure(figsize=(15, 6))
plt.plot(sp500_data.index, sp500_data['Daily_Return'], linewidth=0.8, alpha=0.7)
plt.title('S&P 500 Daily Returns (%)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Daily Return (%)', fontsize=12)
plt.axhline(y=0, color='r', linestyle='--', linewidth=1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of daily returns
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sp500_data['Daily_Return'].hist(bins=50, edgecolor='black')
plt.title('Distribution of Daily Returns', fontsize=14)
plt.xlabel('Daily Return (%)')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sp500_data['Daily_Return'].plot(kind='box')
plt.title('Box Plot of Daily Returns', fontsize=14)
plt.ylabel('Daily Return (%)')

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Extract close prices for forecasting
close_prices = get_close_prices(sp500_data)

# Handle any missing values
if close_prices.isnull().sum() > 0:
    close_prices = fill_missing_values(close_prices, method='ffill')
    print(f"Filled {close_prices.isnull().sum()} missing values")

print(f"\nClose prices shape: {close_prices.shape}")
print(f"Date range: {close_prices.index[0]} to {close_prices.index[-1]}")

In [ ]:
# Split data into train and test sets (80-20 split)
train_data, test_data = train_test_split_timeseries(close_prices, test_size=0.2)

print(f"Training set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")
print(f"\nTraining period: {train_data.index[0]} to {train_data.index[-1]}")
print(f"Test period: {test_data.index[0]} to {test_data.index[-1]}")

In [ ]:
# Visualize train-test split
plt.figure(figsize=(15, 6))
plt.plot(train_data.index, train_data.values, label='Training Data', linewidth=1.5)
plt.plot(test_data.index, test_data.values, label='Test Data', linewidth=1.5)
plt.title('Train-Test Split of S&P 500 Close Prices', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Close Price (USD)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Model 1: Moving Average

In [ ]:
# Train Moving Average model
ma_model = MovingAverageModel(window=7)
ma_model.fit(train_data)

# Make predictions
ma_predictions = ma_model.predict(steps=len(test_data))

# Evaluate
ma_results = evaluate_forecast(test_data.values, ma_predictions)
print("\nMoving Average Model Results:")
print_evaluation_results(ma_results)

## 6. Model 2: Linear Regression

In [ ]:
# Train Linear Regression model
lr_model = LinearRegressionModel()
lr_model.fit(train_data)

# Make predictions
lr_predictions = lr_model.predict(steps=len(test_data))

# Evaluate
lr_results = evaluate_forecast(test_data.values, lr_predictions)
print("\nLinear Regression Model Results:")
print_evaluation_results(lr_results)

## 7. Model 3: ARIMA

In [ ]:
# Train ARIMA model
print("Training ARIMA model...")
arima_model = ARIMAModel(order=(5, 1, 0))
arima_model.fit(train_data)

# Make predictions
arima_predictions = arima_model.predict(steps=len(test_data))

# Evaluate
arima_results = evaluate_forecast(test_data.values, arima_predictions)
print("\nARIMA Model Results:")
print_evaluation_results(arima_results)

In [ ]:
# Display ARIMA model summary
print(arima_model.get_summary())

## 8. Model 4: SARIMAX

In [ ]:
# Train SARIMAX model
print("Training SARIMAX model...")
sarimax_model = SARIMAXModel(order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
sarimax_model.fit(train_data)

# Make predictions
sarimax_predictions = sarimax_model.predict(steps=len(test_data))

# Evaluate
sarimax_results = evaluate_forecast(test_data.values, sarimax_predictions)
print("\nSARIMAX Model Results:")
print_evaluation_results(sarimax_results)

## 9. Model 5: Exponential Smoothing

In [ ]:
# Train Exponential Smoothing model
print("Training Exponential Smoothing model...")
es_model = ExponentialSmoothingModel(trend='add', seasonal=None, seasonal_periods=12)
es_model.fit(train_data)

# Make predictions
es_predictions = es_model.predict(steps=len(test_data))

# Evaluate
es_results = evaluate_forecast(test_data.values, es_predictions)
print("\nExponential Smoothing Model Results:")
print_evaluation_results(es_results)

## 10. Model Comparison

In [ ]:
# Compare all models
all_results = {
    'Moving Average': ma_results,
    'Linear Regression': lr_results,
    'ARIMA': arima_results,
    'SARIMAX': sarimax_results,
    'Exponential Smoothing': es_results
}

comparison_df = compare_models(all_results)
print("\nModel Comparison:")
print(comparison_df)

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

metrics = ['RMSE', 'MAE', 'MAPE', 'R²']
for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    comparison_df[metric].plot(kind='bar', ax=ax, color='skyblue', edgecolor='black')
    ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xlabel('Model', fontsize=12)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Find the best model based on RMSE
best_model = comparison_df['RMSE'].idxmin()
print(f"\nBest performing model based on RMSE: {best_model}")
print(f"RMSE: {comparison_df.loc[best_model, 'RMSE']:.4f}")

## 11. Visualize Predictions

In [ ]:
# Plot all predictions together
plt.figure(figsize=(15, 8))

# Plot actual values
plt.plot(train_data.index, train_data.values, label='Training Data', linewidth=1.5, alpha=0.7)
plt.plot(test_data.index, test_data.values, label='Actual Test Data', linewidth=2, color='black')

# Plot predictions
plt.plot(test_data.index, ma_predictions, label='Moving Average', linewidth=1.5, linestyle='--')
plt.plot(test_data.index, lr_predictions, label='Linear Regression', linewidth=1.5, linestyle='--')
plt.plot(test_data.index, arima_predictions, label='ARIMA', linewidth=1.5, linestyle='--')
plt.plot(test_data.index, sarimax_predictions, label='SARIMAX', linewidth=1.5, linestyle='--')
plt.plot(test_data.index, es_predictions, label='Exponential Smoothing', linewidth=1.5, linestyle='--')

plt.title('S&P 500 Price Predictions - All Models', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Close Price (USD)', fontsize=12)
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot individual model predictions in separate subplots
fig, axes = plt.subplots(3, 2, figsize=(18, 15))
axes = axes.flatten()

predictions = [
    ('Moving Average', ma_predictions),
    ('Linear Regression', lr_predictions),
    ('ARIMA', arima_predictions),
    ('SARIMAX', sarimax_predictions),
    ('Exponential Smoothing', es_predictions)
]

for idx, (model_name, preds) in enumerate(predictions):
    ax = axes[idx]
    
    # Plot training data
    ax.plot(train_data.index, train_data.values, label='Training', linewidth=1, alpha=0.5)
    
    # Plot actual test data
    ax.plot(test_data.index, test_data.values, label='Actual', linewidth=2, color='black')
    
    # Plot predictions
    ax.plot(test_data.index, preds, label='Predicted', linewidth=2, linestyle='--', color='red')
    
    ax.set_title(f'{model_name} Predictions', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.set_ylabel('Price (USD)', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Remove the last empty subplot
fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

## 12. Future Forecasting

In [ ]:
# Use the best model to forecast future prices
# Re-train on full dataset
print("Re-training ARIMA model on full dataset for future forecasting...")
future_model = ARIMAModel(order=(5, 1, 0))
future_model.fit(close_prices)

# Forecast next 30 days
future_steps = 30
future_predictions = future_model.predict(steps=future_steps)

# Create future dates
last_date = close_prices.index[-1]
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=future_steps, freq='D')

print(f"\nForecasted prices for next {future_steps} days:")
future_df = pd.DataFrame({
    'Date': future_dates,
    'Predicted_Price': future_predictions
})
print(future_df.head(10))

In [ ]:
# Visualize future forecast
plt.figure(figsize=(15, 6))

# Plot historical data (last 90 days)
recent_data = close_prices[-90:]
plt.plot(recent_data.index, recent_data.values, label='Historical Data', linewidth=2)

# Plot future predictions
plt.plot(future_dates, future_predictions, label=f'Forecast ({future_steps} days)', 
         linewidth=2, linestyle='--', color='red', marker='o', markersize=4)

plt.title(f'S&P 500 Price Forecast - Next {future_steps} Days', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Close Price (USD)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Summary and Conclusions

In [ ]:
print("="*70)
print("SUMMARY OF TIME SERIES FORECASTING ON S&P 500")
print("="*70)
print(f"\nDataset Period: {close_prices.index[0]} to {close_prices.index[-1]}")
print(f"Total Data Points: {len(close_prices)}")
print(f"Training Set Size: {len(train_data)}")
print(f"Test Set Size: {len(test_data)}")
print(f"\nModels Tested: {len(all_results)}")
print("\nModel Performance (sorted by RMSE):")
print(comparison_df.sort_values('RMSE'))
print(f"\nBest Model: {best_model}")
print(f"Best RMSE: {comparison_df.loc[best_model, 'RMSE']:.4f}")
print(f"Best MAE: {comparison_df.loc[best_model, 'MAE']:.4f}")
print(f"Best MAPE: {comparison_df.loc[best_model, 'MAPE']:.4f}%")
print("="*70)

## Key Insights:

1. **Data Quality**: The S&P 500 data shows good quality with minimal missing values.

2. **Model Performance**: Different models have varying levels of accuracy. Traditional statistical models like ARIMA and SARIMAX often perform well for financial time series.

3. **Model Selection**: The choice of model depends on:
   - Data characteristics (trend, seasonality)
   - Forecast horizon
   - Computational resources
   - Required accuracy

4. **Future Work**:
   - Try deep learning models (LSTM, GRU)
   - Incorporate external features (sentiment, macro indicators)
   - Experiment with ensemble methods
   - Perform hyperparameter optimization

## Disclaimer:
This analysis is for educational purposes only. Stock market predictions are inherently uncertain and should not be used as the sole basis for investment decisions.